In [1]:
import os
import pandas as pd
from pathlib import Path

os.chdir(Path(os.getcwd()).parent)
os.getcwd()

'c:\\Users\\Tomas\\Desktop\\Thesis Stuff\\Survival_Analysis_Thesis\\Coding'

# Data Cleaning and Saving

Produces the interim datasets the rest of the project consumes, via
`DataCleaner.get_clean_data`. Two flavours per segment:

- **Personal | Professional RAW** - merged, one-day users removed, end-year imputed, segmented by user
  type. No cutoff or inactivity filtering. Used for the distributional exploration
  in `01`.

- **All Users RAW** - merged, one-day users removed, end-year imputed. No segmentation. No cutoff or inactivity filtering. Used for the distributional exploration
  in `01`.

- **Personal | Professional FILTERED** - additionally cutoff-filtered, NaN-metadata removed, and truncated
  at the first churn event (adds `churn_adjusted_date`). Used for the interval
  grid and the survival models.


In [2]:
from src.constants import paths_to_files_and_folders as const
from src.data_cleaning import DataCleaner
from src.constants.segments import PERSONAL, PROFESSIONAL
from src.constants.cleaning import DEFAULT_HHI_THRESHOLD, DEFAULT_CAR_SHARE_ABS, DEFAULT_CAR_SHARE_FRACTION

In [3]:
activity_df = pd.read_csv(const.PATH_TO_RAW_ACTIVITY_DATA_1000)
vehicle_df  = pd.read_csv(const.PATH_TO_RAW_VEHICLE_DATA_1000)
data_cleaner = DataCleaner(activity_df, vehicle_df)

# Decided churn threshold (see interval / gap analysis in notebook 01)
CHURN_THRESHOLD_DAYS_PERSONAL = PERSONAL.churn_threshold_days
CHURN_THRESHOLD_DAYS_PROFESSIONAL = PROFESSIONAL.churn_threshold_days

# Shared split criteria
HHI_THRESHOLD       = DEFAULT_HHI_THRESHOLD
CAR_SHARE_ABS       = DEFAULT_CAR_SHARE_ABS
CAR_SHARE_FRACTION  = DEFAULT_CAR_SHARE_FRACTION



## All Users - RAW

In [4]:
merged_all_raw = data_cleaner.get_clean_data(
    merge_data_frames=True,
    filter_nan_cols=None,
    filter_by_user_type=False,
    # return_personal_use_users=True,          
    filter_early_churners=True,
    transform_vehicle_end_year_to_present=True,
    inverse_hhi_threshold=HHI_THRESHOLD,
    car_share_threshold_abs=CAR_SHARE_ABS,
    car_share_threshold_fraction=CAR_SHARE_FRACTION,
    save_file_to=const.PATH_TO_INTERIM_DATA / "merged_all_users_raw.csv",
)

_Step 1_
Rows before merging: 567953
Rows after merging: 567953

Removing user_ids that aren't present in vehicle dataset
Rows before user filtering: 567953
Rows after user filtering: 567352

Removing duplicate rows
Rows before deduplication: 567352
Rows after deduplication: 560583

_Step 2_
Transforming missing vehicle_end_year values to current year
Current year used: 2026
Missing vehicle_end_year values filled: 41626

_Step 3_
Number of early churners removed: 198
Rows before early-churner filtering: 560583
Rows after early-churner filtering: 557378
Rows removed: 3205


Cleaning complete
Final rows: 557378
Final unique users: 2557

_Step 4_
Saving File to C:\Users\Tomas\Desktop\Thesis Stuff\Survival_Analysis_Thesis\Coding\Data\interim\merged_all_users_raw.csv


## Personal - RAW


In [5]:
personal_raw = data_cleaner.get_clean_data(
    merge_data_frames=True,
    filter_nan_cols=None,
    filter_by_user_type=True,
    return_personal_use_users=True,          # personal
    filter_early_churners=True,
    transform_vehicle_end_year_to_present=True,
    inverse_hhi_threshold=HHI_THRESHOLD,
    car_share_threshold_abs=CAR_SHARE_ABS,
    car_share_threshold_fraction=CAR_SHARE_FRACTION,
    save_file_to=const.PATH_TO_INTERIM_DATA / "personal_users_raw.csv",
)

_Step 1_
Rows before merging: 567953
Rows after merging: 567953

Removing user_ids that aren't present in vehicle dataset
Rows before user filtering: 567953
Rows after user filtering: 567352

Removing duplicate rows
Rows before deduplication: 567352
Rows after deduplication: 560583

_Step 2_
Transforming missing vehicle_end_year values to current year
Current year used: 2026
Missing vehicle_end_year values filled: 41626

_Step 3_
Number of early churners removed: 198
Rows before early-churner filtering: 560583
Rows after early-churner filtering: 557378
Rows removed: 3205

_Step 4_
Rows before user type filtering: 557378
Filtering by personal users!
Rows after user type filtering: 176434


Cleaning complete
Final rows: 176434
Final unique users: 2350

_Step 5_
Saving File to C:\Users\Tomas\Desktop\Thesis Stuff\Survival_Analysis_Thesis\Coding\Data\interim\personal_users_raw.csv


## Professional - RAW


In [6]:
professional_raw = data_cleaner.get_clean_data(
    merge_data_frames=True,
    filter_nan_cols=None,
    filter_by_user_type=True,
    return_personal_use_users=False,         # professional
    filter_early_churners=True,
    transform_vehicle_end_year_to_present=True,
    inverse_hhi_threshold=HHI_THRESHOLD,
    car_share_threshold_abs=CAR_SHARE_ABS,
    car_share_threshold_fraction=CAR_SHARE_FRACTION,
    save_file_to=const.PATH_TO_INTERIM_DATA / "professional_users_raw.csv",
)

_Step 1_
Rows before merging: 567953
Rows after merging: 567953

Removing user_ids that aren't present in vehicle dataset
Rows before user filtering: 567953
Rows after user filtering: 567352

Removing duplicate rows
Rows before deduplication: 567352
Rows after deduplication: 560583

_Step 2_
Transforming missing vehicle_end_year values to current year
Current year used: 2026
Missing vehicle_end_year values filled: 41626

_Step 3_
Number of early churners removed: 198
Rows before early-churner filtering: 560583
Rows after early-churner filtering: 557378
Rows removed: 3205

_Step 4_
Rows before user type filtering: 557378
Filtering by professional users!
Rows after user type filtering: 63375


Cleaning complete
Final rows: 63375
Final unique users: 207

_Step 5_
Saving File to C:\Users\Tomas\Desktop\Thesis Stuff\Survival_Analysis_Thesis\Coding\Data\interim\professional_users_raw.csv


## Personal - FILTERED

Adds cutoff filtering, NaN-metadata removal, and inactivity truncation on top of
the raw pipeline.


In [7]:
personal_filtered = data_cleaner.get_clean_data(
    merge_data_frames=True,
    filter_inactivity=True,
    filter_nan_cols=None,
    filter_by_user_type=True,
    return_personal_use_users=True,          # personal
    filter_early_churners=True,
    filter_by_set_cutoff_date=True,
    transform_vehicle_end_year_to_present=True,
    filter_nan_vehicle_metadata=True,
    threshold_value=CHURN_THRESHOLD_DAYS_PERSONAL,
    inverse_hhi_threshold=HHI_THRESHOLD,
    car_share_threshold_abs=CAR_SHARE_ABS,
    car_share_threshold_fraction=CAR_SHARE_FRACTION,
    save_file_to=const.PATH_TO_INTERIM_DATA / "personal_users_filtered.csv",
)

_Step 1_
Rows before merging: 567953
Rows after merging: 567953

Removing user_ids that aren't present in vehicle dataset
Rows before user filtering: 567953
Rows after user filtering: 567352

Removing duplicate rows
Rows before deduplication: 567352
Rows after deduplication: 560583

_Step 2_
Transforming missing vehicle_end_year values to current year
Current year used: 2026
Missing vehicle_end_year values filled: 41626

_Step 3_
Number of early churners removed: 198
Rows before early-churner filtering: 560583
Rows after early-churner filtering: 557378
Rows removed: 3205

_Step 4_
Rows before set cutoff date filtering: 557378
Rows after set cutoff date filtering: 556252
Rows removed: 1126

_Step 5_
Rows before user type filtering: 556252
Filtering by personal users!
Rows after user type filtering: 175880

_Step 6_
Rows before vehicle metadata filtering: 175880
Rows after vehicle metadata filtering: 165670
Rows removed: 10210

_Step 7_
Filtering activity after inactivity threshold: 160 

## Professional - FILTERED


In [8]:
professional_filtered = data_cleaner.get_clean_data(
    merge_data_frames=True,
    filter_inactivity=True,
    filter_nan_cols=None,
    filter_by_user_type=True,
    return_personal_use_users=False,         # professional
    filter_early_churners=True,
    filter_by_set_cutoff_date=True,
    transform_vehicle_end_year_to_present=True,
    filter_nan_vehicle_metadata=True,
    threshold_value=CHURN_THRESHOLD_DAYS_PROFESSIONAL, 
    inverse_hhi_threshold=HHI_THRESHOLD,
    car_share_threshold_abs=CAR_SHARE_ABS,
    car_share_threshold_fraction=CAR_SHARE_FRACTION,
    save_file_to=const.PATH_TO_INTERIM_DATA / "professional_users_filtered.csv",
)

_Step 1_
Rows before merging: 567953
Rows after merging: 567953

Removing user_ids that aren't present in vehicle dataset
Rows before user filtering: 567953
Rows after user filtering: 567352

Removing duplicate rows
Rows before deduplication: 567352
Rows after deduplication: 560583

_Step 2_
Transforming missing vehicle_end_year values to current year
Current year used: 2026
Missing vehicle_end_year values filled: 41626

_Step 3_
Number of early churners removed: 198
Rows before early-churner filtering: 560583
Rows after early-churner filtering: 557378
Rows removed: 3205

_Step 4_
Rows before set cutoff date filtering: 557378
Rows after set cutoff date filtering: 557350
Rows removed: 28

_Step 5_
Rows before user type filtering: 557350
Filtering by professional users!
Rows after user type filtering: 63375

_Step 6_
Rows before vehicle metadata filtering: 63375
Rows after vehicle metadata filtering: 60466
Rows removed: 2909

_Step 7_
Filtering activity after inactivity threshold: 80 day